!! POT SER QUE FALLI SI L'ORDINADOR NO TÉ PROU MEMÒRIA RAM

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from utils.data import get_hits, z_score

pd.set_option("display.max_columns", None)

In [ ]:
# Convert to parquet
# tsv_file = "sexism2 Recording3.tsv"
tsv_file = "sexism4 Data export"

tsv_path = Path("../data", tsv_file + ".tsv")
parquet_file = tsv_path.with_suffix(".parquet")
if not os.path.exists(parquet_file):
    print("creant parquet i tsv amb només primera línia")
    df = pd.read_csv(tsv_path, sep="\t", low_memory=False)
    df.to_parquet(parquet_file, index=False)
    os.rename(tsv_path, tsv_path.with_stem(tsv_file + "_original"))
    df.iloc[:1].to_csv(tsv_path, sep="\t", index=False)

In [ ]:
# Read without unwanted columns
cols = pd.read_csv(tsv_path, sep="\t", low_memory=False, nrows=0).columns
aoi_size = [col for col in cols if "AOI size" in col]
aoi_hit = [col for col in cols if "AOI hit" in col]
calibration_cols = [col for col in cols if "calibration" in col] + [col for col in cols if "validation" in col]
unwanted_cols = aoi_size + [
    "Client area position X (DACSpx)",
    "Client area position Y (DACSpx)",
    "Viewport position X",
    "Viewport position Y",
    "Viewport width",
    "Viewport height",
    "Full page width",
    "Full page height",
    "Mouse position X",
    "Mouse position Y",
    "Sensor",
    "Project name",
    "Export date",
    "Recording name",
    "Recording date",
    "Recording date UTC",
    "Recording start time",
    "Recording start time UTC",
    "Recording duration",
    "Timeline name",
    "Recording Fixation filter name",
    "Recording software version",
    "Recording resolution height",
    "Recording resolution width",
    "Recording monitor latency",
    "Presented Media width",
    "Presented Media height",
    "Presented Media position X (DACSpx)",
    "Presented Media position Y (DACSpx)",
    "Original Media width",
    "Original Media height",
]
cols_to_keep = [col for col in cols if col not in unwanted_cols]

column_mapping = {
    "Recording timestamp": "timestamp",
    "Gaze point X": "gaze_x",
    "Gaze point Y": "gaze_y",
    "Eye movement type": "event_type",  # Fixation, Saccade, etc.
    "Eye movement type index": "event_index",
    "Fixation point X": "fix_x",
    "Fixation point Y": "fix_y",
    "Gaze event duration": "duration",
    "Participant name": "participant",
}
raw_df = pd.read_parquet(parquet_file, engine="fastparquet", columns=cols_to_keep).rename(
    columns={k: v for k, v in column_mapping.items()}
)
calibration_df = raw_df[["participant"] + calibration_cols].drop_duplicates()
participants = raw_df["participant"].unique().tolist()

raw_df = raw_df.drop(calibration_cols, axis=1)
# raw_df = raw_df.dropna(subset=["gaze_x", "gaze_y"])
# drop [nan, 'Unclassified', 'EyesNotFound']
df = raw_df[raw_df["event_type"].isin(("Fixation", "Saccade"))].copy()

In [ ]:
raw_df

In [ ]:
text_dfs = process_all_texts(df)

In [ ]:
def split_by_text(df):
    """
    Split the main DataFrame into a list of DataFrames,
    one per text (between TextStart and TextEnd events).
    """
    text_dfs = []

    # Find indices of TextStart and TextEnd
    start_indices = df.index[df["Event"] == "TextStart"].tolist()
    end_indices = df.index[df["Event"] == "TextEnd"].tolist()

    # Validate pairing
    if len(start_indices) != len(end_indices):
        print(f"Warning: {len(start_indices)} TextStart vs {len(end_indices)} TextEnd events")
        # Pair up what we can
        n_pairs = min(len(start_indices), len(end_indices))
        start_indices = start_indices[:n_pairs]
        end_indices = end_indices[:n_pairs]

    for i, (start, end) in enumerate(zip(start_indices, end_indices)):
        text_df = df.loc[start:end].copy()  # inclusive of both markers
        text_df.attrs["text_index"] = i  # store metadata
        text_dfs.append(text_df)
        print(f"Text {i}: rows {start}-{end} ({len(text_df)} samples)")

    return text_dfs


def get_aoi_columns(df):
    """Get all AOI hit columns."""
    return [col for col in df.columns if col.startswith("AOI hit")]


def drop_irrelevant_aois(text_df):
    """
    Drop AOI columns that are entirely NaN for this text.
    Keep columns that have 0s and 1s (relevant to this text).
    NaN = AOI doesn't belong to this text.
    0   = AOI belongs to this text but wasn't hit on that sample.
    1   = AOI was hit.
    """
    aoi_cols = get_aoi_columns(text_df)
    # Columns that are ALL NaN → not relevant for this text
    cols_to_drop = [col for col in aoi_cols if text_df[col].isna().all()]
    text_df = text_df.drop(columns=cols_to_drop)
    return text_df


def process_all_texts(df):
    """
    Full pipeline: split by text, then clean AOI columns per text.
    Returns a list of clean DataFrames.
    """
    text_dfs = split_by_text(df)

    cleaned_dfs = []
    for text_df in text_dfs:
        cleaned = drop_irrelevant_aois(text_df)
        cleaned_dfs.append(cleaned)

    return cleaned_dfs


def extract_text_id_from_aois(text_df):
    """
    Extract the text/stimulus ID from AOI column names.
    e.g., 'AOI hit [291 - Ya]' → text ID is 291
    """
    aoi_cols = get_aoi_columns(text_df)
    text_ids = set()
    for col in aoi_cols:
        # Extract number between '[' and ' -'
        try:
            text_id = col.split("[")[1].split(" -")[0].strip()
            text_ids.add(text_id)
        except IndexError:
            pass
    return text_ids

In [ ]:
# Access individual texts
for i, tdf in enumerate(text_dfs):
    text_ids = extract_text_id_from_aois(tdf)
    aoi_cols = get_aoi_columns(tdf)
    print(f"\nText {i} | Stimulus ID(s): {text_ids} | " f"{len(tdf)} rows | {len(aoi_cols)} AOIs")

    # Example: see AOI hit distribution for this text
    for col in aoi_cols:
        hits = tdf[col].sum()
        print(f"    {col}: {int(hits)} hits")
    break

In [ ]:
# Now use each text_df with the regression detection from before
# e.g.:
# for tdf in text_dfs:
#     fixations = extract_fixations(tdf)
#     regressions = detect_regressions(fixations)# Access individual texts
for i, tdf in enumerate(text_dfs):
    text_ids = extract_text_id_from_aois(tdf)
    aoi_cols = get_aoi_columns(tdf)
    print(f"\nText {i} | Stimulus ID(s): {text_ids} | " f"{len(tdf)} rows | {len(aoi_cols)} AOIs")

    # Example: see AOI hit distribution for this text
    for col in aoi_cols:
        hits = tdf[col].sum()
        print(f"    {col}: {int(hits)} hits")

# Now use each text_df with the regression detection from before
# e.g.:
# for tdf in text_dfs:
#     fixations = extract_fixations(tdf)
#     regressions = detect_regressions(fixations)

In [ ]:
df["AOI hit [369 - ¿no?]"].value_counts(dropna=False)

In [ ]:
raw_df.columns

In [ ]:
raw_df.Event.unique()

In [ ]:
[c for c in df.columns if "duration" in c]

In [ ]:
df[~df.Event.isna()]

In [ ]:
raw_df[~raw_df.Event.isna()]

In [ ]:
df[["gaze_x", "gaze_y"]]

In [ ]:
# df[df[aoi_hit[0]] != 0]

In [ ]:
aoi = aoi_hit[3]
print(aoi)
df[df[aoi] != 0]

In [ ]:
# df[cols].head()

In [ ]:
# hits to int
df[aoi_hit] = df[aoi_hit].fillna(0).astype(int)

df = z_score(df, "Pupil diameter filtered", "participant")

In [ ]:
calibration_df

In [ ]:
text_hits = get_hits(df, aoi_hit, "participant", normalize=True)

In [ ]:
text_hits["vivir"]

In [ ]:
# todo: is there a "text" column?, else -> create it
# -> unique [text, participant] to check if they did all the texts.

In [ ]:
texts = {t.split(" - ")[1].rsplit("]", 1)[0] for t in aoi_hit}
# texts = [t.split("[")[1].split()[0] for t in aoi_hit] # old?
text = list(texts)[2]
print(text)
plt.scatter(text_hits[text]["hits"], text_hits[text]["z_pupil"])
plt.show()

In [ ]:
# Eye movement type	Eye movement event duration

In [ ]:
df[df.Event.isna()]

In [ ]:
# df[~df['AOI hit [Text - tu]'].isna()]

In [ ]:
def extract_fixations(df):
    """Extract fixation events from Tobii data."""
    if "event_type" in df.columns:
        # Tobii already classified events
        fixations = df[df["event_type"] == "Fixation"].copy()

        # Group by fixation index to get one row per fixation
        fixations = (
            fixations.groupby("event_index")
            .agg(
                {
                    "timestamp": "first",
                    "fix_x": "mean",
                    "fix_y": "mean",
                    "gaze_x": "mean",
                    "gaze_y": "mean",
                    "duration": "first",
                }
            )
            .reset_index()
        )

        fixations["x"] = fixations["fix_x"].fillna(fixations["gaze_x"])
        fixations["y"] = fixations["fix_y"].fillna(fixations["gaze_y"])
    else:
        # Use raw gaze points as-is (consider applying I-VT or I-DT filter)
        fixations = df[["timestamp", "gaze_x", "gaze_y"]].copy()
        fixations.rename(columns={"gaze_x": "x", "gaze_y": "y"}, inplace=True)

    fixations = fixations.sort_values("timestamp").reset_index(drop=True)
    return fixations

In [ ]:
fixations = extract_fixations(df)

In [ ]:
# ============================================
# 3. SIMPLE I-DT FIXATION DETECTOR (if needed)
# ============================================
def idt_fixation_detector(df, dispersion_threshold=50, duration_threshold=100):
    """
    Identify fixations using Dispersion-Threshold Identification (I-DT).
    - dispersion_threshold: max pixel spread for a fixation
    - duration_threshold: min duration in ms
    """
    points = df[["timestamp", "gaze_x", "gaze_y"]].dropna().values
    fixations = []
    i = 0

    while i < len(points):
        j = i + 1
        while j < len(points):
            window = points[i : j + 1]
            dispersion = window[:, 1].max() - window[:, 1].min() + window[:, 2].max() - window[:, 2].min()
            duration = window[-1, 0] - window[0, 0]

            if dispersion > dispersion_threshold:
                if duration >= duration_threshold:
                    fixations.append(
                        {
                            "timestamp": window[0, 0],
                            "x": window[:, 1].mean(),
                            "y": window[:, 2].mean(),
                            "duration": duration,
                        }
                    )
                    i = j
                    break
                else:
                    i += 1
                    break
            j += 1
        else:
            # End of data
            window = points[i:]
            duration = window[-1, 0] - window[0, 0]
            if duration >= duration_threshold:
                fixations.append(
                    {
                        "timestamp": window[0, 0],
                        "x": window[:, 1].mean(),
                        "y": window[:, 2].mean(),
                        "duration": duration,
                    }
                )
            break

    return pd.DataFrame(fixations)

In [ ]:
# ============================================
# 4. DETECT REGRESSIONS
# ============================================
def detect_regressions(fixations, reading_direction="ltr", line_height_threshold=50, min_regression_distance=30):
    """
    Detect regression saccades from fixation sequence.

    Parameters:
    -----------
    fixations : DataFrame with columns ['timestamp', 'x', 'y']
    reading_direction : 'ltr' (left-to-right) or 'rtl' (right-to-left)
    line_height_threshold : max Y difference to consider same line (pixels)
    min_regression_distance : min backward distance to count as regression (pixels)

    Returns:
    --------
    DataFrame of regressions with details
    """
    regressions = []

    for i in range(1, len(fixations)):
        prev = fixations.iloc[i - 1]
        curr = fixations.iloc[i]

        dx = curr["x"] - prev["x"]
        dy = curr["y"] - prev["y"]
        same_line = abs(dy) < line_height_threshold

        is_regression = False
        regression_type = None

        if same_line:
            # Within-line regression
            if reading_direction == "ltr" and dx < -min_regression_distance:
                is_regression = True
                regression_type = "within-line"
            elif reading_direction == "rtl" and dx > min_regression_distance:
                is_regression = True
                regression_type = "within-line"
        else:
            # Between-line regression (going back to a previous line)
            if dy < -line_height_threshold:
                is_regression = True
                regression_type = "between-line"

        if is_regression:
            regressions.append(
                {
                    "fixation_index": i,
                    "timestamp_from": prev["timestamp"],
                    "timestamp_to": curr["timestamp"],
                    "from_x": prev["x"],
                    "from_y": prev["y"],
                    "to_x": curr["x"],
                    "to_y": curr["y"],
                    "distance_x": dx,
                    "distance_y": dy,
                    "saccade_length": np.sqrt(dx**2 + dy**2),
                    "regression_type": regression_type,
                }
            )

    return pd.DataFrame(regressions)

In [ ]:
# ============================================
# 5. COMPUTE REGRESSION METRICS
# ============================================
def compute_regression_metrics(fixations, regressions):
    """Compute summary statistics about regressions."""
    total_fixations = len(fixations)
    total_saccades = total_fixations - 1
    total_regressions = len(regressions)

    metrics = {
        "total_fixations": total_fixations,
        "total_saccades": total_saccades,
        "total_regressions": total_regressions,
        "regression_rate": total_regressions / total_saccades if total_saccades > 0 else 0,
        "within_line_regressions": (
            len(regressions[regressions["regression_type"] == "within-line"]) if len(regressions) > 0 else 0
        ),
        "between_line_regressions": (
            len(regressions[regressions["regression_type"] == "between-line"]) if len(regressions) > 0 else 0
        ),
        "mean_regression_distance": regressions["saccade_length"].mean() if len(regressions) > 0 else 0,
        "max_regression_distance": regressions["saccade_length"].max() if len(regressions) > 0 else 0,
    }

    return metrics


# ============================================
# 6. AOI-BASED REGRESSIONS (for defined regions)
# ============================================
def detect_aoi_regressions(fixations, aois):
    """
    Detect regressions between Areas of Interest.

    aois: list of dicts with {'name', 'x_min', 'x_max', 'y_min', 'y_max', 'order'}
          where 'order' is the expected reading order (1, 2, 3...)
    """

    def get_aoi(x, y):
        for aoi in aois:
            if aoi["x_min"] <= x <= aoi["x_max"] and aoi["y_min"] <= y <= aoi["y_max"]:
                return aoi["name"], aoi["order"]
        return None, None

    # Assign AOIs to fixations
    fixations = fixations.copy()
    fixations["aoi_name"], fixations["aoi_order"] = zip(*fixations.apply(lambda r: get_aoi(r["x"], r["y"]), axis=1))

    # Find regressions (going to a lower-order AOI)
    aoi_regressions = []
    for i in range(1, len(fixations)):
        prev_order = fixations.iloc[i - 1]["aoi_order"]
        curr_order = fixations.iloc[i]["aoi_order"]

        if prev_order is not None and curr_order is not None:
            if curr_order < prev_order:
                aoi_regressions.append(
                    {
                        "fixation_index": i,
                        "from_aoi": fixations.iloc[i - 1]["aoi_name"],
                        "to_aoi": fixations.iloc[i]["aoi_name"],
                        "from_order": prev_order,
                        "to_order": curr_order,
                    }
                )

    return pd.DataFrame(aoi_regressions)

In [ ]:
# ============================================
# 7. VISUALIZATION
# ============================================
def plot_scanpath_with_regressions(fixations, regressions, screen_size=(1920, 1080), title="Scanpath with Regressions"):
    """Visualize fixations and highlight regressions."""
    fig, ax = plt.subplots(1, 1, figsize=(14, 8))

    # Plot all fixations
    ax.scatter(
        fixations["x"],
        fixations["y"],
        s=fixations.get("duration", pd.Series([50] * len(fixations))) / 5,
        c="steelblue",
        alpha=0.6,
        zorder=3,
        label="Fixations",
    )

    # Draw all saccades (light gray)
    for i in range(1, len(fixations)):
        ax.plot(
            [fixations.iloc[i - 1]["x"], fixations.iloc[i]["x"]],
            [fixations.iloc[i - 1]["y"], fixations.iloc[i]["y"]],
            "gray",
            alpha=0.3,
            linewidth=0.8,
        )

    # Highlight regressions in red
    for _, reg in regressions.iterrows():
        ax.annotate(
            "",
            xy=(reg["to_x"], reg["to_y"]),
            xytext=(reg["from_x"], reg["from_y"]),
            arrowprops=dict(arrowstyle="->", color="red", lw=2, alpha=0.7),
        )

    ax.set_xlim(0, screen_size[0])
    ax.set_ylim(screen_size[1], 0)  # Invert Y axis (screen coordinates)
    ax.set_xlabel("X (pixels)")
    ax.set_ylabel("Y (pixels)")
    ax.set_title(title)
    ax.legend()
    plt.tight_layout()
    plt.savefig("regressions_plot.png", dpi=150)
    plt.show()


def plot_regression_heatmap(regressions, screen_size=(1920, 1080)):
    """Heatmap of regression landing points."""
    fig, ax = plt.subplots(figsize=(14, 8))

    if len(regressions) > 0:
        hb = ax.hexbin(regressions["to_x"], regressions["to_y"], gridsize=20, cmap="Reds", mincnt=1)
        plt.colorbar(hb, ax=ax, label="Regression count")

    ax.set_xlim(0, screen_size[0])
    ax.set_ylim(screen_size[1], 0)
    ax.set_title("Regression Landing Points Heatmap")
    plt.tight_layout()
    plt.savefig("regression_heatmap.png", dpi=150)
    plt.show()

In [ ]:
# ============================================
# 8. MAIN PIPELINE
# ============================================
if __name__ == "__main__":
    # --- Load data ---
    filepath = "tobii_export.tsv"  # Change to your file
    df = load_tobii_data(filepath)
    df = preprocess_tobii_data(df)

    # --- Extract fixations ---
    # Option A: Use Tobii's built-in classification
    fixations = extract_fixations(df)

    # Option B: Use custom I-DT detector (uncomment if needed)
    # fixations = idt_fixation_detector(df, dispersion_threshold=50, duration_threshold=100)

    print(f"Total fixations: {len(fixations)}")

    # --- Detect regressions ---
    regressions = detect_regressions(
        fixations, reading_direction="ltr", line_height_threshold=50, min_regression_distance=30
    )

    print(f"\nRegressions found: {len(regressions)}")
    print(regressions.head(10))

    # --- Metrics ---
    metrics = compute_regression_metrics(fixations, regressions)
    print("\n--- Regression Metrics ---")
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

    # --- Optional: AOI-based regressions ---
    # aois = [
    #     {'name': 'Word1', 'x_min': 100, 'x_max': 200, 'y_min': 300, 'y_max': 350, 'order': 1},
    #     {'name': 'Word2', 'x_min': 210, 'x_max': 350, 'y_min': 300, 'y_max': 350, 'order': 2},
    #     {'name': 'Word3', 'x_min': 360, 'x_max': 500, 'y_min': 300, 'y_max': 350, 'order': 3},
    # ]
    # aoi_regs = detect_aoi_regressions(fixations, aois)
    # print(aoi_regs)

    # --- Visualize ---
    plot_scanpath_with_regressions(fixations, regressions)
    plot_regression_heatmap(regressions)

    # --- Export results ---
    regressions.to_csv("regressions_output.csv", index=False)
    print("\nResults saved to regressions_output.csv")